# 10 - PySpark: Fundamentos

Apache Spark para processamento distribuido de dados.

## 1. SparkSession

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window

spark = SparkSession.builder.master('local[2]').appName('datalab-fundamentos').getOrCreate()
spark.sparkContext.setLogLevel('WARN')
print(f'Spark {spark.version} - {spark.sparkContext.master}')

## 2. Schema Explicito

In [ ]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType, DateType
import datetime

schema = StructType([
    StructField('id', IntegerType(), False),
    StructField('nome', StringType(), False),
    StructField('depto', StringType(), False),
    StructField('salario', DoubleType(), False),
    StructField('data_admissao', DateType(), True),
])

dados = [
    (1, 'Ana', 'TI', 9500.0, datetime.date(2020, 1, 15)),
    (2, 'Joao', 'RH', 7200.0, datetime.date(2019, 6, 1)),
    (3, 'Maria', 'TI', 11000.0, datetime.date(2021, 3, 10)),
    (4, 'Pedro', 'Financeiro', 8800.0, datetime.date(2018, 11, 20)),
    (5, 'Lucia', 'RH', 6500.0, datetime.date(2022, 8, 5)),
    (6, 'Carlos', 'Vendas', 10500.0, datetime.date(2020, 7, 12)),
    (7, 'Fernanda', 'TI', 12000.0, datetime.date(2017, 4, 3)),
    (8, 'Ricardo', 'Vendas', 7800.0, datetime.date(2021, 9, 18)),
]

df = spark.createDataFrame(dados, schema=schema)
df.show()

## 3. Select e Filter

In [ ]:
df.select('nome', 'depto', 'salario').filter(F.col('salario') > 8000).show()

## 4. GroupBy e Agg

In [ ]:
df.groupBy('depto').agg(
    F.count('*').alias('qtd'),
    F.round(F.avg('salario'), 2).alias('media'),
    F.max('salario').alias('maximo'),
).orderBy(F.desc('media')).show()

## 5. Join

In [ ]:
deptos_df = spark.createDataFrame([
    ('TI', 'Sao Paulo'),
    ('RH', 'Rio'),
    ('Financeiro', 'Belo Horizonte'),
    ('Vendas', 'Curitiba'),
], ['depto', 'localizacao'])

df.join(deptos_df, on='depto', how='left').show()

## 6. Window Functions

In [ ]:
w_dept = Window.partitionBy('depto').orderBy(F.desc('salario'))

df.withColumn('rank_depto', F.rank().over(w_dept))\
  .withColumn('soma_depto', F.sum('salario').over(w_dept)).show()

## 7. Explain

In [ ]:
df.groupBy('depto').agg(F.avg('salario')).explain()

## 8. Exercicio

Crie uma coluna `faixa_salario` com `F.when` que classifique: <8000=Junior, 8000-11000=Pleno, >11000=Senior.

## Conclusao

PySpark segue a mesma logica de pandas mas com lazy evaluation.

In [ ]:
spark.stop()